# MTU Notebook Workflow

Run Mapbox Tileset Uploader from a notebook with one editable configuration cell and reusable helper logic.

Default behavior is **dry run** so you can validate conversion and geometry checks without uploading.

## Required Inputs (Edit First)

Edit these values at the top of Cell 4 before running:

- `SOURCE_FILE`: local input file path (most important for file mode).
- `SOURCE_MODE`: `'file'` or `'url'`.
- `SOURCE_URL`: remote source URL when using URL mode.
- `MAPBOX_ACCESS_TOKEN` and `MAPBOX_USERNAME` (from env or Colab Secrets).
- `TILESET_ID`, `TILESET_NAME`, and `LAYER_NAME`.

## Fast Start

1. If needed, run Cell 2 to install `mtu` in the current kernel (`INSTALL_MTU = True`).
2. Run Cell 3 to confirm `mtu` is available.
3. Edit the **USER INPUTS** block at the top of Cell 4.
4. Run all remaining cells top to bottom.
5. For a real upload, set `DRY_RUN = False` in Cell 4 and run Cell 8 again.

You do not need to edit any cells outside Cell 4 (except optional `INSTALL_MTU` in Cell 2).

## Cell Guide

- Cell 1: Overview, instructions, and reference notes (this section).
- Cell 2: Optional package install helper for the active kernel.
- Cell 3: Dependency check for `mtu` in the active kernel.
- Cell 4: User inputs and configuration assembly (`SETTINGS`).
- Cell 5: Quick pointer to top-level documentation.
- Cell 6: Imports required by helper and execution cells.
- Cell 7: Reusable helper functions (preflight checks, preview, builder helpers).
- Cell 8: One-click run cell (uses `SETTINGS`, performs upload/dry-run, prints results).
- Cell 9: Quick pointer to top-level documentation.

## Source And Format Notes

- Local file mode: set `SOURCE_MODE = 'file'` and update `SOURCE_FILE`.
- URL mode: set `SOURCE_MODE = 'url'` and update `SOURCE_URL`.
- Keep `FORMAT_HINT = None` to auto-detect format unless you need to force one.

Supported input formats: GeoJSON, TopoJSON, Shapefile ZIP/SHP, GeoPackage, KML/KMZ, FlatGeobuf, GeoParquet, GPX.

## Limits And Cautions

- Required token scopes: `tilesets:read`, `tilesets:write`, `tilesets:list`.
- URL-restricted tokens may fail with `403 Forbidden` in CLI/notebook flows.
- Typical Mapbox zoom range is `0-22`; notebook defaults are `4-8`.
- Workflow default upload cap is 1 GB (`APP_DEFAULT_UPLOAD_CAP_GB`).
- You can opt into Mapbox 20 GB per-file cap with `USE_MAPBOX_FULL_UPLOAD_CAP = True`.
- Remaining Mapbox CU cannot be queried from this notebook.
- Higher zoom ranges and heavier datasets may increase billable usage.

## How To Read Output (Cell 8)

- `success`: whether the workflow completed without error.
- `dry_run`: whether the run was simulation or real upload.
- `tileset_id`: target tileset identifier used for the run.
- `steps`: ordered list of pipeline steps executed.
- `warnings`: non-fatal issues to review before production runs.

## Colab: Load Variables And Source Files

### Credentials In Colab (recommended: Secrets)

1. Click the key icon in the left sidebar (`Secrets`).
2. Add `MAPBOX_ACCESS_TOKEN` and `MAPBOX_USERNAME`.
3. Keep them enabled for this notebook.
4. Cell 4 auto-detects Colab and reads these values.

### Alternative: environment variables in a cell

```python
import os
os.environ['MAPBOX_ACCESS_TOKEN'] = 'your-token'
os.environ['MAPBOX_USERNAME'] = 'your-username'
```

### Load source files in Colab

- Upload from local machine:

```python
from google.colab import files
uploaded = files.upload()
```

Then set `SOURCE_MODE = 'file'` and `SOURCE_FILE = Path('/content/<your-file-name>')` in Cell 4.

- Or use URL source:
  Set `SOURCE_MODE = 'url'` and `SOURCE_URL = 'https://...'` in Cell 4.

In [ ]:
# Optional install helper for this notebook kernel.
# Set INSTALL_MTU = True only if Cell 3 says mtu is missing.
import subprocess
import sys

INSTALL_MTU = True
INSTALL_TARGET = "mtu"  # or a pinned version, e.g. "mtu==0.1.0"

if INSTALL_MTU:
    print(f"Installing {INSTALL_TARGET} into: {sys.executable}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", INSTALL_TARGET])
    print("Install complete. Re-run Cell 3 to verify.")
else:
    print("Install skipped. Set INSTALL_MTU = True to install mtu in this kernel.")

# Notes:
# - For local source instead of PyPI package, run in terminal:
#   python -m pip install -e .


In [ ]:
# Dependency check (especially useful in Colab)
import importlib.util

has_mtu = importlib.util.find_spec('mtu') is not None

if has_mtu:
    print('mtu package is available in this kernel.')
else:
    print('mtu package is NOT installed in this kernel.')
    print('Install via Cell 2 (set INSTALL_MTU = True), then re-run Cell 3.')
    print('Alternative for local source in terminal: python -m pip install -e .')
    print('Restart the runtime/kernel if import errors persist after install.')

In [ ]:
# Configuration only: edit values in the USER INPUTS block below, then run remaining cells unchanged.
import os
from pathlib import Path
from typing import Literal, TypedDict, cast

# =============================
# USER INPUTS (EDIT THESE)
# =============================

# Auth (fallbacks to env/Colab when left blank).
MAPBOX_ACCESS_TOKEN = os.environ.get('MAPBOX_ACCESS_TOKEN', '')
MAPBOX_USERNAME = os.environ.get('MAPBOX_USERNAME', '')

# Run behavior.
DRY_RUN = True

# Source settings (most users only need to edit SOURCE_FILE).
SOURCE_MODE_RAW = os.environ.get('MTU_SOURCE_MODE', 'file').strip().lower()  # 'file' or 'url'
SOURCE_FILE = Path(r'...\data.geojson')
SOURCE_URL = 'https://example.com/data.geojson'
FORMAT_HINT: str | None = None  # None enables auto-detect

# Tileset settings.
TILESET_ID = 'demo-tileset-id'
TILESET_NAME = 'Demo Tileset'
LAYER_NAME = 'data'
MIN_ZOOM = 4
MAX_ZOOM = 8

# Upload limits and optional planning.
MAPBOX_ZOOM_MIN = 0
MAPBOX_ZOOM_MAX = 22
APP_DEFAULT_UPLOAD_CAP_GB = 1.0
MAPBOX_FULL_UPLOAD_CAP_GB = 20.0
USE_MAPBOX_FULL_UPLOAD_CAP = False
CAPACITY_LIMIT_MB = 0.0
CAPACITY_USED_MB = 0.0

class Settings(TypedDict):
    mapbox_access_token: str
    mapbox_username: str
    dry_run: bool
    source_mode: Literal['file', 'url']
    source_file: Path
    source_url: str
    format_hint: str | None
    tileset_id: str
    tileset_name: str
    layer_name: str
    min_zoom: int
    max_zoom: int
    mapbox_zoom_min: int
    mapbox_zoom_max: int
    app_default_upload_cap_gb: float
    mapbox_full_upload_cap_gb: float
    use_mapbox_full_upload_cap: bool
    capacity_limit_mb: float
    capacity_used_mb: float

# Colab secrets can override env variables automatically.
_token_from_colab: str = ''
_username_from_colab: str = ''
try:
    from google.colab import userdata  # type: ignore

    _token_from_colab = str(userdata.get('MAPBOX_ACCESS_TOKEN') or '')
    _username_from_colab = str(userdata.get('MAPBOX_USERNAME') or '')
except Exception:
    _token_from_colab = ''
    _username_from_colab = ''

if SOURCE_MODE_RAW not in {'file', 'url'}:
    raise ValueError("SOURCE_MODE_RAW must be 'file' or 'url'")
SOURCE_MODE = cast(Literal['file', 'url'], SOURCE_MODE_RAW)

SETTINGS: Settings = {
    'mapbox_access_token': _token_from_colab or MAPBOX_ACCESS_TOKEN,
    'mapbox_username': _username_from_colab or MAPBOX_USERNAME,
    'dry_run': DRY_RUN,
    'source_mode': SOURCE_MODE,
    'source_file': SOURCE_FILE,
    'source_url': SOURCE_URL,
    'format_hint': FORMAT_HINT,
    'tileset_id': TILESET_ID,
    'tileset_name': TILESET_NAME,
    'layer_name': LAYER_NAME,
    'min_zoom': MIN_ZOOM,
    'max_zoom': MAX_ZOOM,
    'mapbox_zoom_min': MAPBOX_ZOOM_MIN,
    'mapbox_zoom_max': MAPBOX_ZOOM_MAX,
    'app_default_upload_cap_gb': APP_DEFAULT_UPLOAD_CAP_GB,
    'mapbox_full_upload_cap_gb': MAPBOX_FULL_UPLOAD_CAP_GB,
    'use_mapbox_full_upload_cap': USE_MAPBOX_FULL_UPLOAD_CAP,
    'capacity_limit_mb': CAPACITY_LIMIT_MB,
    'capacity_used_mb': CAPACITY_USED_MB,
}

print('Configuration loaded. Update USER INPUTS at top of this cell only.')
print(f'dry_run={DRY_RUN}, source_mode={SOURCE_MODE}')
print(f'source_file={SOURCE_FILE}')
print(f'source_url={SOURCE_URL}')
print(f'format_hint={FORMAT_HINT}')
print(f'zoom={MIN_ZOOM}-{MAX_ZOOM} (Mapbox typical {MAPBOX_ZOOM_MIN}-{MAPBOX_ZOOM_MAX})')
print(
    'upload_cap_mode=' + (
        'Mapbox full 20 GB' if USE_MAPBOX_FULL_UPLOAD_CAP else 'App default 1 GB'
    )
)
print(f'MAPBOX_ACCESS_TOKEN set: {bool(SETTINGS["mapbox_access_token"])}')
print(f'MAPBOX_USERNAME set: {bool(SETTINGS["mapbox_username"])}')

## Reference

> Key documentation, limits, and supported formats are now at the top in Cell 1.

If you need to change behavior, update `SETTINGS` in Cell 4 only.

In [ ]:
import json
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

from mtu.uploader import TilesetConfig, TilesetUploader

In [ ]:
# Reusable helpers. No user edits needed in this cell.
def build_tileset_config(settings: Settings) -> TilesetConfig:
    return TilesetConfig(
        tileset_id=settings['tileset_id'],
        tileset_name=settings['tileset_name'],
        layer_name=settings['layer_name'],
        min_zoom=settings['min_zoom'],
        max_zoom=settings['max_zoom'],
    )


def build_uploader(settings: Settings) -> TilesetUploader:
    return TilesetUploader(
        validate_geometry=True,
        use_mapbox_full_upload_cap=settings['use_mapbox_full_upload_cap'],
    )


def get_active_upload_cap_gb(settings: Settings) -> float:
    return (
        settings['mapbox_full_upload_cap_gb']
        if settings['use_mapbox_full_upload_cap']
        else settings['app_default_upload_cap_gb']
    )


def describe_source(settings: Settings) -> None:
    cap_gb = get_active_upload_cap_gb(settings)
    cap_mb = cap_gb * 1024
    print(f'Active upload cap: {cap_gb} GB')

    if settings['source_mode'] == 'url':
        source_url = settings['source_url']
        print(f'URL source selected: {source_url}')
        parsed = urlparse(source_url)
        if not parsed.scheme or not parsed.netloc:
            raise ValueError('SOURCE_URL must be a valid absolute URL when source_mode is url')
        if settings['capacity_limit_mb'] > 0:
            print('Note: local size/capacity projection is skipped for URL sources.')
        return

    source_file = settings['source_file']
    if not source_file.exists():
        print(f'WARNING: file not found: {source_file}')
        return

    size_mb = source_file.stat().st_size / (1024 * 1024)
    print(f'Input size: {size_mb:.2f} MB')
    if size_mb > cap_mb:
        print('WARNING: input file exceeds active upload cap.')

    capacity_limit_mb = settings['capacity_limit_mb']
    if capacity_limit_mb > 0:
        projected = settings['capacity_used_mb'] + size_mb
        print(f'Projected usage: {projected:.2f} / {capacity_limit_mb:.2f} MB')
        if projected > capacity_limit_mb:
            print('WARNING: projected usage exceeds configured capacity limit.')

    if source_file.suffix.lower() not in {'.geojson', '.json'}:
        return

    # Optional map preview for local GeoJSON files using folium.
    try:
        import folium
        from IPython.display import display

        with source_file.open('r', encoding='utf-8') as f:
            geojson_data: Any = json.load(f)

        m = folium.Map(location=[0, 0], zoom_start=settings['min_zoom'], control_scale=True)
        layer = folium.GeoJson(geojson_data, name='source data')
        layer.add_to(m)

        try:
            bounds = layer.get_bounds()
            if bounds and len(bounds) == 2:
                sw, ne = bounds
                if all(v is not None for v in [sw[0], sw[1], ne[0], ne[1]]):
                    m.fit_bounds([[float(sw[0]), float(sw[1])], [float(ne[0]), float(ne[1])]])
        except Exception:
            pass

        display(m)
    except Exception as ex:
        print('Map preview skipped (folium not installed or invalid GeoJSON):', ex)

Active upload cap: 1 GB
Input size: 7.38 MB


In [ ]:
# One-click execution cell: run after updating SETTINGS in Cell 4.
describe_source(SETTINGS)

config = build_tileset_config(SETTINGS)
uploader = build_uploader(SETTINGS)
format_hint: str | None = SETTINGS['format_hint']

if SETTINGS['source_mode'] == 'url':
    result = uploader.upload_from_url(
        url=SETTINGS['source_url'],
        config=config,
        format_hint=format_hint,
        dry_run=SETTINGS['dry_run'],
    )
else:
    result = uploader.upload_from_file(
        file_path=SETTINGS['source_file'],
        config=config,
        format_hint=format_hint,
        dry_run=SETTINGS['dry_run'],
    )

print('success:', result.success)
print('dry_run:', result.dry_run)
print('tileset_id:', result.tileset_id)
print('steps:', result.steps)
print('warnings:', result.warnings)

success: True
dry_run: True
tileset_id: ocha-rosea-1.demo-tileset-id
steps: {'convert': True, 'validate': True}
warnings: []


## Reference

> Real upload and output interpretation notes are documented at the top in Cell 1.

For production runs: set `SETTINGS['dry_run'] = False` in Cell 4, then rerun Cell 8.